In [9]:
import os
import json
from pathlib import Path

import pandas as pd
import textstat

from google import genai

print("Baseline notebook setup successful!")

Baseline notebook setup successful!


In [10]:
import sys
print(sys.executable)

/Users/tkabwe/Documents/Google 2E/Google-2E-faithfulmed-a-multi-agent-system-for-faithful-patient-friendly-medical-explanations/.venv/bin/python


In [11]:
import sys
import pandas as pd
import textstat
from google import genai

print("Python:", sys.version)
print("Environment:", sys.executable)
print("All imports successful!")

Python: 3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:47:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Environment: /Users/tkabwe/Documents/Google 2E/Google-2E-faithfulmed-a-multi-agent-system-for-faithful-patient-friendly-medical-explanations/.venv/bin/python
All imports successful!


In [12]:
# A small example to test our single-Gemini baseline

sample_medical_text = """
The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.
"""

print(sample_medical_text)



The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.



In [13]:
# Fixed instructions for our single-Gemini baseline

BASELINE_PROMPT = """
You are a medical information simplification assistant.

Rewrite the provided medical text as a clear, patient-friendly explanation.

Follow these rules:
1. Use plain, everyday English.
2. Aim for an 8th-grade reading level or below.
3. Explain medical terminology using simple language.
4. Preserve all clinically important information from the original text.
5. Do not invent symptoms, diagnoses, treatments, or medical facts.
6. Do not make recommendations that are not supported by the source text.
7. Do not remove important warnings, measurements, or uncertainty.
8. Return only the rewritten explanation.

Medical text:
{medical_text}
"""

# Insert our first medical example into the prompt
test_prompt = BASELINE_PROMPT.format(
    medical_text=sample_medical_text
)

print(test_prompt)


You are a medical information simplification assistant.

Rewrite the provided medical text as a clear, patient-friendly explanation.

Follow these rules:
1. Use plain, everyday English.
2. Aim for an 8th-grade reading level or below.
3. Explain medical terminology using simple language.
4. Preserve all clinically important information from the original text.
5. Do not invent symptoms, diagnoses, treatments, or medical facts.
6. Do not make recommendations that are not supported by the source text.
7. Do not remove important warnings, measurements, or uncertainty.
8. Return only the rewritten explanation.

Medical text:

The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.




In [14]:
from getpass import getpass
import os
from google import genai

# Ask for the API key without displaying it on screen
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

# Create the Gemini client
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Gemini client initialized!")


Enter your Gemini API key:  ········


Gemini client initialized!


In [16]:
import time
from google.genai import errors

MODEL_NAME = "gemini-3.8-flash"

baseline_output = None

for attempt in range(3):
    try:
        print(f"Attempt {attempt + 1}...")

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=test_prompt,
        )

        baseline_output = response.text
        break

    except errors.ServerError as e:
        if e.code == 503 and attempt < 2:
            print("Gemini is busy. Waiting 15 seconds...")
            time.sleep(15)
        else:
            raise

if baseline_output:
    print("\nORIGINAL MEDICAL TEXT:")
    print(sample_medical_text)

    print("\nGEMINI'S PATIENT-FRIENDLY EXPLANATION:")
    print(baseline_output)

Attempt 1...

ORIGINAL MEDICAL TEXT:

The patient presents with hypertension and elevated blood glucose.
Lifestyle modifications, including dietary changes and regular
physical activity, are recommended to reduce cardiovascular risk.


GEMINI'S PATIENT-FRIENDLY EXPLANATION:
You have high blood pressure (hypertension) and high blood sugar. Making healthy daily changes, such as adjusting your diet and getting regular exercise, is recommended to help lower your risk of heart problems.


In [17]:
from eval_harness import readability_scores, meets_readability_target

scores = readability_scores(baseline_output)

print("Gemini output:")
print(baseline_output)

print("\nReadability scores:")
print(scores)

print("\nMeets 8th-grade target:")
print(meets_readability_target(baseline_output))

Gemini output:
You have high blood pressure (hypertension) and high blood sugar. Making healthy daily changes, such as adjusting your diet and getting regular exercise, is recommended to help lower your risk of heart problems.

Readability scores:
{'flesch_kincaid_grade': 10.511666666666667, 'smog_index': 12.161744961471694, 'jargon_density': 0.30303030303030304, 'word_count': 33}

Meets 8th-grade target:
False


In [18]:
original_scores = readability_scores(sample_medical_text)

print("ORIGINAL TEXT SCORES:")
print(original_scores)

print("\nGEMINI OUTPUT SCORES:")
print(scores)

print(
    "\nGrade-level improvement:",
    round(
        original_scores["flesch_kincaid_grade"]
        - scores["flesch_kincaid_grade"],
        2
    )
)

ORIGINAL TEXT SCORES:
{'flesch_kincaid_grade': 18.59, 'smog_index': 15.903189008614273, 'jargon_density': 0.6666666666666666, 'word_count': 24}

GEMINI OUTPUT SCORES:
{'flesch_kincaid_grade': 10.511666666666667, 'smog_index': 12.161744961471694, 'jargon_density': 0.30303030303030304, 'word_count': 33}

Grade-level improvement: 8.08


In [19]:
result = {
    "example_id": "test_001",
    "source_text": sample_medical_text.strip(),
    "gemini_output": baseline_output.strip(),
    "model": MODEL_NAME,
    "source_grade": original_scores["flesch_kincaid_grade"],
    "output_grade": scores["flesch_kincaid_grade"],
    "grade_improvement": (
        original_scores["flesch_kincaid_grade"]
        - scores["flesch_kincaid_grade"]
    ),
    "smog_index": scores["smog_index"],
    "jargon_density": scores["jargon_density"],
    "word_count": scores["word_count"],
    "meets_grade_8": meets_readability_target(baseline_output),
}

result

{'example_id': 'test_001',
 'source_text': 'The patient presents with hypertension and elevated blood glucose.\nLifestyle modifications, including dietary changes and regular\nphysical activity, are recommended to reduce cardiovascular risk.',
 'gemini_output': 'You have high blood pressure (hypertension) and high blood sugar. Making healthy daily changes, such as adjusting your diet and getting regular exercise, is recommended to help lower your risk of heart problems.',
 'model': 'gemini-3.8-flash',
 'source_grade': 18.59,
 'output_grade': 10.511666666666667,
 'grade_improvement': 8.078333333333333,
 'smog_index': 12.161744961471694,
 'jargon_density': 0.30303030303030304,
 'word_count': 33,
 'meets_grade_8': False}

In [22]:
import time
from google.genai import errors

def run_baseline(example_id, medical_text):
    prompt = BASELINE_PROMPT.format(
        medical_text=medical_text
    )

    # Try up to 3 times if Gemini is temporarily busy
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
            )
            break

        except errors.ServerError as e:
            if e.code == 503 and attempt < 2:
                print(
                    f"Gemini busy. "
                    f"Retrying in 15 seconds... "
                    f"(attempt {attempt + 1}/3)"
                )
                time.sleep(15)
            else:
                raise

    output = response.text.strip()

    source_scores = readability_scores(medical_text)
    output_scores = readability_scores(output)

    return {
        "example_id": example_id,
        "source_text": medical_text.strip(),
        "gemini_output": output,
        "model": MODEL_NAME,
        "source_grade": source_scores["flesch_kincaid_grade"],
        "output_grade": output_scores["flesch_kincaid_grade"],
        "grade_improvement": (
            source_scores["flesch_kincaid_grade"]
            - output_scores["flesch_kincaid_grade"]
        ),
        "smog_index": output_scores["smog_index"],
        "jargon_density": output_scores["jargon_density"],
        "word_count": output_scores["word_count"],
        "meets_grade_8": meets_readability_target(output),
    }

print("Baseline function with retry protection created!")

Baseline function with retry protection created!


In [23]:
test_result = run_baseline(
    example_id="test_002",
    medical_text=sample_medical_text
)

test_result

Gemini busy. Retrying in 15 seconds... (attempt 1/3)
Gemini busy. Retrying in 15 seconds... (attempt 2/3)


{'example_id': 'test_002',
 'source_text': 'The patient presents with hypertension and elevated blood glucose.\nLifestyle modifications, including dietary changes and regular\nphysical activity, are recommended to reduce cardiovascular risk.',
 'gemini_output': 'You have high blood pressure and high blood sugar. Making healthy everyday changes, such as changing what you eat and getting regular physical activity, is recommended to lower your risk of heart problems.',
 'model': 'gemini-3.8-flash',
 'source_grade': 18.59,
 'output_grade': 10.154090909090915,
 'grade_improvement': 8.435909090909085,
 'smog_index': 12.161744961471694,
 'jargon_density': 0.2727272727272727,
 'word_count': 33,
 'meets_grade_8': False}

In [24]:
from pathlib import Path

data_dir = Path("../data")

print("Files available in data/:")
for file in sorted(data_dir.iterdir()):
    print("-", file.name)

Files available in data/:
- .gitkeep
- Confirmation.pdf
- MedAESQA_methods_M1-M30.xlsx
- README.md
- medaesqa_v1.json
- retrieval_docs_medlineplus.jsonl
- retrieval_docs_medquad_medlineplus.jsonl


In [26]:
from pathlib import Path

project_root = Path("..")

search_terms = [
    "baseline",
    "50",
    "PLABA",
    "MTSamples",
    "gold set",
    "frozen"
]

for path in project_root.rglob("*"):
    if path.is_file() and path.suffix in {".py", ".md"}:
        try:
            text = path.read_text(errors="ignore")
            matches = [
                term for term in search_terms
                if term.lower() in text.lower()
            ]

            if matches:
                print(f"{path}: {matches}")
        except Exception:
            pass

../README.md: ['baseline']
../Challenge-Project-Overview.md: ['baseline', '50', 'PLABA', 'MTSamples', 'gold set', 'frozen']
../data/README.md: ['PLABA', 'MTSamples', 'gold set', 'frozen']
../notebooks/dataset_profile.py: ['50']
../.venv/lib/python3.12/site-packages/typing_extensions.py: ['50', 'frozen']
../.venv/lib/python3.12/site-packages/fastjsonschema/draft07.py: ['50']
../.venv/lib/python3.12/site-packages/fastjsonschema/__init__.py: ['50']
../.venv/lib/python3.12/site-packages/fastjsonschema/ref_resolver.py: ['50']
../.venv/lib/python3.12/site-packages/fastjsonschema/draft04.py: ['frozen']
../.venv/lib/python3.12/site-packages/overrides/typing_utils.py: ['frozen']
../.venv/lib/python3.12/site-packages/packaging/tags.py: ['frozen']
../.venv/lib/python3.12/site-packages/packaging/dependency_groups.py: ['50']
../.venv/lib/python3.12/site-packages/packaging/metadata.py: ['frozen']
../.venv/lib/python3.12/site-packages/packaging/version.py: ['frozen']
../.venv/lib/python3.12/site-pack